### Preprocessing data

In [1]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score

# Preprocess the data
initial_data = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/refined-set-csv.csv')
initial_data = initial_data[~initial_data['Protein_FASTA'].str.contains('X')] # remove complexes with ambiguous char in FASTA string
toDrop = ['4yx4', '1laf', '4buq', '1k22', '1hmt', '1utn', '4rux', '1bty']
initial_data = initial_data[~initial_data['PDB_Code'].isin(toDrop)]
initial_data = initial_data.reset_index() # final dataset

# Load protein and ligand embeddings data
protein_t6 = np.load("/dcs/22/u2243582/cs310/seq_embeddings/protein_embeddings_t6.npy")
ligand_t6 = np.load("/dcs/22/u2243582/cs310/seq_embeddings/ligand_embeddings_t6.npy")
protein_t12 = np.load("/dcs/22/u2243582/cs310/seq_embeddings/protein_embeddings_t12.npy")
ligand_t12 = np.load("/dcs/22/u2243582/cs310/seq_embeddings/ligand_embeddings_t12.npy")

/dcs/22/u2243582/.local/lib/python3.9/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [3]:
# Single array of embeddings data
embeddings = np.concatenate((protein_t12, ligand_t12), axis=1) # (5059, 864)

# Scale features and apply PCA before we split into folds!
scaler = StandardScaler()
scaler.fit(embeddings)
embeddings_scaled = scaler.transform(embeddings)
X = pd.DataFrame(embeddings_scaled, index=initial_data.index) # convert to DataFrame

pca = PCA(0.85) # keep 85% of the variance of the data
pca.fit(X)
X_pca = pca.transform(X)
X = pd.DataFrame(X_pca, index=initial_data.index, columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]) # convert back to DataFrame

print("Components:", pca.n_components_ , "Total explained variance:", pca.explained_variance_ratio_.sum())

Components: 65 Total explained variance: 0.85157335


In [4]:
data = pd.merge(X, initial_data, left_index=True, right_index=True)
#data = data.reset_index(drop=True)
#display(data)

### Making non-redundant cross-validation folds from the similarity clusters using CDHIT program

In [46]:
# Run CDHit on indicies.fasta - the indicies are the same just diff PCA values

import json

with open("/dcs/22/u2243582/cs310/nrkf_physicochem_feat/clustered_proteins.json", "r") as json_file:
    clustered = json.load(json_file)
print(len(clustered))

# Create the dictionary clusters = {'obj1': 1, 'obj2': 2, 'obj3': 1, 'obj4': 2, 'obj5': 3, 'obj6': 3}
# key obj is the index of the examples in data, value int is cluster id
clusters = {}
for clusterInd, indList in clustered.items():
    for ind in indList:
        clusters[ind] = clusterInd

# Create the list objects = ['obj1', 'obj2', 'obj3', 'obj4', 'obj5', 'obj6', 'obj1'] where obj is an index in data
objects = list(clusters.keys())

1192


#### https://github.com/foxtrotmike/BioTools/blob/main/NRKFold.py

In [47]:
def NRKFold(E,pc,K = 5, shuffle=True):
    """
    Generate non-redundant K-folds for a dataset where each example involves an object 
    that belongs to a certain cluster. This function ensures that no two folds contain 
    objects from the same cluster and aims to distribute the number of examples 
    approximately equally across all folds.

    This is particularly useful in scenarios where data points can naturally group into 
    clusters (e.g., proteins in bioinformatics), and it is important to avoid having 
    similar examples in both the training and validation sets of a particular fold.

    Parameters
    ----------
    E : List
        A list containing identifiers of objects involved in each example.
    pc : Dictionary
        A dictionary mapping each object to its cluster assignment.
    K : Integer, optional
        The number of folds to create. Default is 5.
    shuffle : Boolean, optional
        Determines whether to shuffle the cluster to fold assignments in different runs.
        Default is True.

    Returns
    -------
    List of lists
        A list where each sublist contains the indices of examples in `objects` that belong to a particular fold.

    Example
    -------
    >>> objects = ['obj1', 'obj2', 'obj3', 'obj4', 'obj5', 'obj6', 'obj1']
    >>> clusters = {'obj1': 1, 'obj2': 2, 'obj3': 1, 'obj4': 2, 'obj5': 3, 'obj6': 3}
    >>> folds = NRKFold(objects, clusters, K=2, shuffle=False)
    >>> print(folds)
    Output might be: [[0, 2, 6], [1, 3, 4, 5]]
    Here, objects 'obj1', 'obj3', and 'obj1' (indices 0, 2, 6) are in one fold, 
    and the rest are in another fold, ensuring no fold has objects from the same cluster.
    """
    e = [pc[str(x)] for x in E] #cluster indices of all proteins in the examples
    c2idx={} #indices of examples of each cluster in e
    for i,x in enumerate(e):
        try: 
            c2idx[x].append(i)
        except:
            c2idx[x]=[i]    
    ce = dict([(c,len(c2idx[c])) for c in c2idx]) #counts of examples of different clusters    
    cF = [0]*K; #counts of examples in each fold
    CF = [[] for _ in range(K)]; #clusters in each fold
    F = [[] for _ in range(K)];#indices of examples in each fold
    keys = list(ce.keys())
    if shuffle:
        random.shuffle(keys)
    for k in keys:
        v = ce[k]
        idx = np.argmin(cF)
        cF[idx]+=v
        CF[idx].append(k) #add cluster to fold
        F[idx].extend(c2idx[k])
    return F

folds = NRKFold(objects, clusters)

In [48]:
# Split the clusters w object list indicies into folds with correct data indicies

indPerFolds = []
for fold in folds: # get the index of examples in each fold
    ind = []
    
    for objInd in fold:
        ind.append(int(objects[objInd]))
    
    indPerFolds.append(ind)
    
# The final NRKfolds:
splits = [
    {
        "train_ix": indPerFolds[0] + indPerFolds[1] + indPerFolds[2] + indPerFolds[3],
        "test_ix": indPerFolds[4]
    },
    {
        "train_ix": indPerFolds[0] + indPerFolds[1] + indPerFolds[2] + indPerFolds[4],
        "test_ix": indPerFolds[3]
    },
    {
       "train_ix": indPerFolds[0] + indPerFolds[1] + indPerFolds[4] + indPerFolds[3],
        "test_ix": indPerFolds[2]
    },
    {
        "train_ix": indPerFolds[0] + indPerFolds[4] + indPerFolds[2] + indPerFolds[3],
        "test_ix": indPerFolds[1]
    },
    {
        "train_ix": indPerFolds[4] + indPerFolds[1] + indPerFolds[2] + indPerFolds[3],
        "test_ix": indPerFolds[0]
    }
]

### Random Forest

In [50]:
# Use modal hyperparameters from baseline nested cv

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:

    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data.iloc[train_ix, :65]
    X_test = data.iloc[test_ix, :65]
    y_train = data.iloc[train_ix, -1]
    y_test = data.iloc[test_ix, -1]
    
    randForest = RandomForestRegressor(n_estimators=300, min_samples_split=10, min_samples_leaf=5, max_features=1.0, max_depth=90)
    randForest.fit(X_train,y_train)
    randForestPred = randForest.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, randForestPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, randForestPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, randForestPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - randForestPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, randForestPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.4749770818605602, Std = 0.042438984158657
Pearson P-value: Mean = 7.988929089950536e-51, Std = 1.1778311051277018e-50
Spearman Correlation: Mean = 0.4650616312539981, Std = 0.04556256454544887
Spearman P-value: Mean = 1.6011284035780203e-44, Std = 3.202256806367016e-44
Mean Absolute Error: Mean = 1.3979568964844598, Std = 0.1332776376498147
Variance of Errors: Mean = 2.8159766271973825, Std = 0.3215225675514937
R2: Mean = 0.16897233486894403, Std = 0.09961191006912087


### SVR

In [49]:
# Use modal hyperparameters from baseline nested cv:
# ...

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:

    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])

    X_train = data.iloc[train_ix, :65]
    X_test = data.iloc[test_ix, :65]
    y_train = data.iloc[train_ix, -1]
    y_test = data.iloc[test_ix, -1]
    
    svRegressor = SVR(kernel='rbf', gamma='scale', epsilon=0.6, C=8)
    svRegressor.fit(X_train, y_train)
    svRegressorPred = svRegressor.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, svRegressorPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, svRegressorPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, svRegressorPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - svRegressorPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, svRegressorPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.59970520451579, Std = 0.03408398025027389
Pearson P-value: Mean = 1.8980780948750142e-80, Std = 3.7961561693477556e-80
Spearman Correlation: Mean = 0.6007274444472196, Std = 0.040434328702544285
Spearman P-value: Mean = 4.1479336863959873e-75, Std = 8.295867372791976e-75
Mean Absolute Error: Mean = 1.2215365423988902, Std = 0.03857716876870067
Variance of Errors: Mean = 2.349454786817645, Std = 0.15328736395376938
R2: Mean = 0.3393521389890511, Std = 0.038413581961445245


### XGBoost

In [51]:
# Use modal hyperparameters from baseline nested cv:
# ...

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:

    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data.iloc[train_ix, :65]
    X_test = data.iloc[test_ix, :65]
    y_train = data.iloc[train_ix, -1]
    y_test = data.iloc[test_ix, -1]
    
    xgReg = XGBRegressor(subsample=0.5, reg_lambda=10, reg_alpha=1, max_depth=9, learning_rate=0.1, gamma=0, colsample_bytree=1)
    xgReg.fit(X_train, y_train)
    xgPred = xgReg.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, xgPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, xgPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, xgPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - xgPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, xgPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.5185718596577057, Std = 0.029013336964106753
Pearson P-value: Mean = 1.569167666858578e-59, Std = 3.1383353331540007e-59
Spearman Correlation: Mean = 0.5121064351425856, Std = 0.031981463466377605
Spearman P-value: Mean = 7.58921663894174e-63, Std = 1.5153722699408104e-62
Mean Absolute Error: Mean = 1.3406359999147976, Std = 0.1226765350531666
Variance of Errors: Mean = 2.650015338098257, Std = 0.30417193588153174
R2: Mean = 0.22713555844159927, Std = 0.08585190817212565
